# Data Cleaning – Dữ liệu giao thông Quy Nhơn

Notebook này được sử dụng để **giải thích và kiểm chứng logic làm sạch dữ liệu**
trước khi áp dụng vào pipeline tự động (MinIO → Clean Zone).

⚠️ Lưu ý:
- Notebook **không chỉnh sửa dữ liệu raw**
- Notebook **không ghi database**
- Notebook chỉ phục vụ **EDA & giải thích trong Report 3**

Pipeline thực tế được triển khai bằng Python script chạy tự động.

In [13]:
import pandas as pd
from minio import Minio
import io

In [14]:
# ================= CONFIG =================
MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "admin123"
BUCKET_NAME = "raw-traffic-data"

SAMPLE_OBJECT = (
    "traffic/incremental/2026-03-03/"
    "traffic_20260303_071752.parquet"
)

# ================= MINIO CLIENT =================
client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

# ================= LOAD SAMPLE FILE =================
response = client.get_object(BUCKET_NAME, SAMPLE_OBJECT)
df_raw = pd.read_parquet(io.BytesIO(response.read()))
response.close()

df_raw.head()

,id,timestamp,location,current_speed_kmh,free_flow_speed_kmh,speed_ratio,traffic_level,confidence
0,21251,2026-02-27 11:36:44,NGÃ 5 ĐỐNG ĐA,52.0,52.0,1.00,THOANG,1.000000
1,21252,2026-02-27 11:36:44,HOÀNG VĂN THỤ - TâY SƠN,50.0,50.0,1.00,THOANG,1.000000
2,21253,2026-02-27 11:36:44,VÒNG XOAY QUẢNG TRƯỜNG NTT,31.0,39.0,0.79,DONG,0.995536
3,21254,2026-02-27 11:36:44,VÒNG XOAY NGUYỄN THÁI HỌC,28.0,38.0,0.74,DONG,0.976996
4,21255,2026-02-27 11:36:44,NGÃ 3 THÁP ĐÔI,27.0,40.0,0.68,DONG,0.969987


## Dữ liệu Raw (Chưa làm sạch)

Dữ liệu raw được crawl và lưu trữ trực tiếp vào MinIO ở định dạng Parquet.

Đặc điểm:
- Có thể chứa giá trị thiếu (null)
- Có thể có timestamp sai định dạng
- Một số record có độ tin cậy (confidence) thấp
- Không phù hợp để đưa trực tiếp vào mô hình Machine Learning

Vì vậy, cần một bước **Data Cleaning** trước khi modeling.

In [15]:
# Không chỉnh sửa df_raw
df_clean = df_raw.copy()

## Nguyên tắc làm sạch dữ liệu

Quá trình clean tuân theo các nguyên tắc:

1. **Raw data là bất biến** (không chỉnh sửa dữ liệu gốc)
2. Chỉ giữ các record có:
   - Tốc độ hợp lệ
   - Free-flow speed > 0
   - Timestamp hợp lệ
   - Độ tin cậy (confidence) > 0.9
3. Không tạo lại feature đã được tính từ trước
4. Dữ liệu sau clean phải sẵn sàng cho Machine Learning

In [16]:
df_clean = df_clean[
    df_clean["current_speed_kmh"].notna() &
    df_clean["free_flow_speed_kmh"].notna() &
    (df_clean["free_flow_speed_kmh"] > 0)
]

### Lọc theo độ tin cậy (Confidence)

Chỉ giữ các record có `confidence > 0.9`.

Lý do:
- Confidence thấp thường đến từ dữ liệu GPS nhiễu
- Dữ liệu không đáng tin sẽ làm giảm chất lượng mô hình
- Ngưỡng 0.9 được chọn để ưu tiên **độ chính xác hơn số lượng**

In [17]:
df_clean = df_clean[
    df_clean["confidence"].notna() &
    (df_clean["confidence"] > 0.9)
]

In [18]:
df_clean["timestamp"] = pd.to_datetime(
    df_clean["timestamp"],
    errors="coerce"
)

df_clean = df_clean[df_clean["timestamp"].notna()]

In [19]:
df_clean = df_clean[
    [
        "id",
        "timestamp",
        "location",
        "current_speed_kmh",
        "free_flow_speed_kmh",
        "speed_ratio",
        "traffic_level",
        "confidence"
    ]
]

df_clean.head()

,id,timestamp,location,current_speed_kmh,free_flow_speed_kmh,speed_ratio,traffic_level,confidence
0,21251,2026-02-27 11:36:44,NGÃ 5 ĐỐNG ĐA,52.0,52.0,1.00,THOANG,1.000000
1,21252,2026-02-27 11:36:44,HOÀNG VĂN THỤ - TâY SƠN,50.0,50.0,1.00,THOANG,1.000000
2,21253,2026-02-27 11:36:44,VÒNG XOAY QUẢNG TRƯỜNG NTT,31.0,39.0,0.79,DONG,0.995536
3,21254,2026-02-27 11:36:44,VÒNG XOAY NGUYỄN THÁI HỌC,28.0,38.0,0.74,DONG,0.976996
4,21255,2026-02-27 11:36:44,NGÃ 3 THÁP ĐÔI,27.0,40.0,0.68,DONG,0.969987


## So sánh số lượng dữ liệu

Notebook này giúp kiểm chứng:
- Bao nhiêu dữ liệu bị loại bỏ
- Ảnh hưởng của các rule làm sạch

In [20]:
print("Số dòng raw:", len(df_raw))
print("Số dòng sau clean:", len(df_clean))

Số dòng raw: 5000
Số dòng sau clean: 4846


## Kết luận

- Notebook này chứng minh logic làm sạch dữ liệu
- Logic này được triển khai tự động trong pipeline production
- Dữ liệu sau clean có độ tin cậy cao, phù hợp cho bài toán
  dự đoán tình trạng tắc đường tại Quy Nhơn (6h–22h)